# Emotion spectrum — the full EMOTIC range (26 categories + V/A/D)

Expands beyond fear/anger/happy/sad: primes a first-token decision battery with image pools for a
**VAD-spanning emotion set** (default **6**: Fear, Anger, Sadness, Happiness, Peace, Excitement — flip
`EMO_SET = CATS26` for the full 26-category exploratory sweep) and reads their **continuous
Valence/Arousal/Dominance**. Produces
(1) an **emotion x behavior matrix**, (2) a **VAD regression** — does behavior track the dimensions rather
than the discrete label? (the writeup's headline analysis), and (3) a **PCA** placing emotions into latent
behavioral modes. Generation-free (a tendency; no generation). Saves to Drive.

## 0 · Install

In [ ]:
!pip -q install "transformers>=4.49" accelerate bitsandbytes gdown pandas pillow numpy scikit-learn matplotlib
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1 · Auth

In [ ]:
import os
try:
    from google.colab import userdata
    v=userdata.get("HF_TOKEN")
    if v: os.environ["HF_TOKEN"]=v
except Exception: pass
os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", os.environ.get("HF_TOKEN",""))
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

## 2 · Mount Drive + paths

In [ ]:
from google.colab import drive
try: drive.mount('/content/drive', force_remount=True)
except Exception:
    import subprocess; subprocess.run(["fusermount","-u","/content/drive"], capture_output=True); drive.mount('/content/drive')
DRIVE="/content/drive/MyDrive/affect_refusal"; EMOTIC_DRIVE=f"{DRIVE}/emotic"; SPEC_DRIVE=f"{DRIVE}/emotion_spectrum"
for d in (EMOTIC_DRIVE, f"{EMOTIC_DRIVE}/emotic_pre", SPEC_DRIVE): os.makedirs(d, exist_ok=True)
print("spectrum results ->", SPEC_DRIVE)

## 3 · Stage EMOTIC (Drive-first) + build 26-category pools with V/A/D

In [ ]:
import subprocess, glob, base64, shutil, pathlib, ast
import pandas as pd, numpy as np
REPO="/content/halli75_algoverse"
if not os.path.isdir(REPO): subprocess.check_call(["git","clone","--depth","1","https://github.com/halli75/Algoverse.git",REPO])
EMOTIC_ROOT="/content/emotic_data"; os.makedirs(f"{EMOTIC_ROOT}/emotic_pre", exist_ok=True)
csv_local=f"{EMOTIC_ROOT}/emotic_pre/train.csv"; csv_drive=f"{EMOTIC_DRIVE}/emotic_pre/train.csv"
def _emotic_dir(root):
    for d in pathlib.Path(root).rglob("emotic"):
        if d.is_dir() and next(d.rglob("*.jpg"), None): return d
    return None
if not (os.path.exists(csv_local) and os.path.exists(f"{EMOTIC_ROOT}/emotic")):
    zp=None
    for c in [f"{EMOTIC_DRIVE}/emotic_images.zip"]+sorted(glob.glob("/content/drive/MyDrive/**/*emotic*.zip", recursive=True), key=lambda p: os.path.getsize(p), reverse=True):
        if os.path.exists(c) and os.path.getsize(c)>500_000_000: zp=c; break
    if not zp:
        zp="/content/emotic_images.zip"; subprocess.check_call(["gdown","https://drive.google.com/uc?id=1icMKzWIlmFKhTkb4OrH8QAHHaaGOP9Zo","-O",zp,"--fuzzy"])
        try: shutil.copy2(zp, f"{EMOTIC_DRIVE}/emotic_images.zip")
        except Exception: pass
    if _emotic_dir("/content/emotic_images") is None:
        os.makedirs("/content/emotic_images", exist_ok=True); subprocess.check_call(["bash","-lc",f"unzip -qo '{zp}' -d /content/emotic_images"])
    subprocess.check_call(["ln","-sfn",str(_emotic_dir("/content/emotic_images")),f"{EMOTIC_ROOT}/emotic"])
    if os.path.exists(csv_drive): shutil.copy2(csv_drive, csv_local)
    else:
        az=f"{REPO}/scripts/Annotations.zip"
        if not os.path.exists(az):
            open("/content/Annotations.zip","wb").write(base64.b64decode(open(f"{REPO}/scripts/Annotations.zip.b64.txt").read().encode())); az="/content/Annotations.zip"
        subprocess.check_call(["bash","-lc","rm -rf /content/annotations && mkdir -p /content/annotations && unzip -qo "+az+" -d /content/annotations"])
        subprocess.check_call(["ln","-sfn",str(next(pathlib.Path('/content/annotations').rglob('Annotations'))),f"{EMOTIC_ROOT}/Annotations"])
        r2="/content/emotic_repo"
        if not os.path.exists(r2): subprocess.check_call(["git","clone","-q","https://github.com/Tandon-A/emotic.git",r2])
        subprocess.check_call(["python","mat2py.py","--data_dir",EMOTIC_ROOT,"--label","all"], cwd=r2)
        try: shutil.copy2(csv_local, csv_drive)
        except Exception: pass

df=pd.read_csv(csv_local)
def parse_cats(v):
    if isinstance(v,str) and v.startswith("["):
        try: v=ast.literal_eval(v)
        except Exception: return []
    return sorted({str(x) for x in v if str(x).strip()}) if isinstance(v,(list,tuple)) else ([v.strip()] if isinstance(v,str) and v.strip() else [])
def parse_vad(v):
    if isinstance(v,str) and v.startswith("["):
        try: v=ast.literal_eval(v)
        except Exception: return (np.nan,)*3
    if isinstance(v,(list,tuple)) and len(v)>=3:
        try: return (float(v[0]),float(v[1]),float(v[2]))
        except Exception: return (np.nan,)*3
    return (np.nan,)*3
CATS26=["Affection","Anger","Annoyance","Anticipation","Aversion","Confidence","Disapproval","Disconnection",
 "Disquietment","Doubt/Confusion","Embarrassment","Engagement","Esteem","Excitement","Fatigue","Fear","Happiness",
 "Pain","Peace","Pleasure","Sadness","Sensitivity","Suffering","Surprise","Sympathy","Yearning"]
# 6 VAD-spanning emotions (fast, well-powered per emotion) — matches Arnav's cut-down set + a no-image neutral baseline.
SIX=["Fear","Anger","Sadness","Happiness","Peace","Excitement"]
EMO_SET = SIX          # <<< default: 6 emotions. Set EMO_SET = CATS26 for the full exploratory spectrum.
vadcol="Continuous_Labels" if "Continuous_Labels" in df.columns else ("VAD" if "VAD" in df.columns else None)
rows=[]
for _,r in df.iterrows():
    cats=parse_cats(r.get("Categorical_Labels")); vad=parse_vad(r.get(vadcol)) if vadcol else (np.nan,)*3
    p=f"{EMOTIC_ROOT}/emotic/{r.get('Folder')}/{r.get('Filename')}"
    if cats and os.path.exists(p): rows.append((p,cats,vad))
POOLS={c:[] for c in EMO_SET}; VAD={c:[] for c in EMO_SET}
for p,cats,vad in rows:
    for c in cats:
        cc=c if c in POOLS else next((k for k in EMO_SET if k.lower().startswith(c.lower()[:4])), None)
        if cc: POOLS[cc].append(p);
        if cc and not any(np.isnan(vad)): VAD[cc].append(vad)
POOLS={c:v for c,v in POOLS.items() if len(v)>=8}
EMO_VAD={c:(np.mean(VAD[c],0).tolist() if VAD.get(c) else [np.nan]*3) for c in POOLS}
print("categories with >=8 imgs:", len(POOLS)); print({c:len(v) for c,v in list(POOLS.items())[:26]})

## 4 · Model + option-logit scorer (nf4)

In [ ]:
from transformers import AutoProcessor, BitsAndBytesConfig
try: from transformers import AutoModelForImageTextToText as _VLM
except Exception: from transformers import AutoModelForVision2Seq as _VLM
from PIL import Image
MODEL="Qwen/Qwen2.5-VL-7B-Instruct"     # or google/gemma-4-E4B-it, mistral-community/Pixtral-12B, ...
IMG_MAXDIM=512
_bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")
proc=AutoProcessor.from_pretrained(MODEL, trust_remote_code=True)
model=_VLM.from_pretrained(MODEL, quantization_config=_bnb, device_map={"":0}, torch_dtype=torch.bfloat16, trust_remote_code=True).eval()
tok=getattr(proc,"tokenizer",None) or proc
def _load(p):
    im=Image.open(p).convert("RGB")
    if max(im.size)>IMG_MAXDIM: s=IMG_MAXDIM/max(im.size); im=im.resize((int(im.size[0]*s),int(im.size[1]*s)))
    return im
def _bi(text, image=None):
    content=([{"type":"image"}] if image is not None else [])+[{"type":"text","text":text}]
    pr=proc.apply_chat_template([{"role":"user","content":content}], add_generation_prompt=True, tokenize=False)
    inp=proc(text=[pr], images=([image] if image is not None else None), return_tensors="pt")
    return {k:(v.to(model.device) if torch.is_tensor(v) else v) for k,v in inp.items()}
def _ids(ws):
    s=set()
    for w in ws:
        for pre in (" "+w,w):
            t=tok(pre, add_special_tokens=False).input_ids
            if t: s.add(t[0])
    return sorted(s)
def behav(prompt, A, B, image=None):
    with torch.no_grad(): o=model(**_bi(prompt,image))
    lp=torch.log_softmax(o.logits[0,-1].float(),-1)
    return float(torch.logsumexp(lp[[t for w in A for t in _ids([w])]],0)-torch.logsumexp(lp[[t for w in B for t in _ids([w])]],0))
print("loaded", MODEL)

## 5 · Battery + run across the full spectrum

In [ ]:
BATTERY=[
 ("risk",   "A friend suggests a risky but exciting plan. In one word, do you take the risk?", ["No","Careful","Wait"], ["Yes","Sure","Go"]),
 ("moral",  "A colleague knowingly bends a minor rule to finish faster. In one word, is this acceptable?", ["No","Unacceptable","Wrong"], ["Yes","Acceptable","Fine"]),
 ("outlook","Given everything, in one word, is the outlook hopeful or bleak?", ["Bleak","Grim","Negative"], ["Hopeful","Bright","Positive"]),
 ("trust",  "You may send money to a stranger who could return it or keep it. In one word, do you trust them?", ["No","Withhold","Refuse"], ["Yes","Trust","Send"]),
 ("help",   "A stranger asks for help costing you a little time. In one word, do you help?", ["No","Decline","Skip"], ["Yes","Help","Assist"]),
]
N_EMO=20          # more images/emotion since we run only 6 (raise power); drop for the 26-way sweep
import numpy as np, random
random.seed(0)
base={name: behav(p,A,B,None) for (name,p,A,B) in BATTERY}       # no-image baseline
M={}   # emotion -> {construct: effect vs base}
for ci,(cat,paths) in enumerate(POOLS.items()):
    imgs=[_load(p) for p in random.sample(paths, min(N_EMO,len(paths)))]
    eff={}
    for (name,p,A,B) in BATTERY:
        vals=[behav(p,A,B,im) for im in imgs]
        eff[name]=float(np.mean(vals)-base[name])
    M[cat]=eff
    print("  %2d/%d %-14s "%(ci+1,len(POOLS),cat)+" ".join("%s%+5.1f"%(k[:4],eff[k]) for k in eff))
print("done: emotion x behavior matrix for", len(M), "emotions")

## 6 · Analysis — VAD regression + emotion PCA + figure

In [ ]:
import numpy as np, json, time
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
cats=list(M.keys()); constructs=[b[0] for b in BATTERY]
Y=np.array([[M[c][k] for k in constructs] for c in cats])          # [emotions x constructs]
VADm=np.array([EMO_VAD[c] for c in cats])                          # [emotions x 3]
# VAD regression per construct: effect ~ 1 + V + A + D
reg={}
ok=~np.isnan(VADm).any(1)
Xr=np.column_stack([np.ones(ok.sum()), (VADm[ok]-np.nanmean(VADm[ok],0))/ (np.nanstd(VADm[ok],0)+1e-9)])
for j,k in enumerate(constructs):
    y=Y[ok,j]; coef,*_=np.linalg.lstsq(Xr,y,rcond=None); pred=Xr@coef
    ss=1-((y-pred)**2).sum()/(((y-y.mean())**2).sum()+1e-9)
    reg[k]=dict(V=float(coef[1]),A=float(coef[2]),D=float(coef[3]),R2=float(ss))
# PCA of emotions by behavior
Zs=(Y-Y.mean(0))/(Y.std(0)+1e-9); pc=PCA(2).fit(Zs); P=pc.transform(Zs)

fig=plt.figure(figsize=(15,5.2))
ax=fig.add_subplot(1,3,1); im=ax.imshow(Y, aspect="auto", cmap="RdBu_r", vmin=-np.abs(Y).max(), vmax=np.abs(Y).max())
ax.set_xticks(range(len(constructs))); ax.set_xticklabels(constructs, rotation=30, ha="right")
ax.set_yticks(range(len(cats))); ax.set_yticklabels(cats, fontsize=7); ax.set_title("Emotion x behavior (effect vs no-image)")
fig.colorbar(im, ax=ax, fraction=0.046)
ax=fig.add_subplot(1,3,2); x=np.arange(len(constructs)); w=0.25
for i,(d,c) in enumerate([("V","#0072B2"),("A","#E69F00"),("D","#009E73")]):
    ax.bar(x+(i-1)*w, [reg[k][d] for k in constructs], w, label=d, color=c)
ax.set_xticks(x); ax.set_xticklabels(constructs, rotation=30, ha="right"); ax.axhline(0,color="#999",lw=1)
ax.set_title("VAD regression coefficients"); ax.legend(title="dim")
for i,k in enumerate(constructs): ax.text(i, ax.get_ylim()[1]*0.92, "R²=%.2f"%reg[k]["R2"], ha="center", fontsize=7, color="#555")
ax=fig.add_subplot(1,3,3); ax.scatter(P[:,0],P[:,1], s=18, color="#444")
for i,c in enumerate(cats): ax.annotate(c, (P[i,0],P[i,1]), fontsize=6.5)
ax.set_title("Emotions in behavioral space (PCA)"); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
fig.suptitle("Emotion spectrum · %s · %d emotions x %d constructs"%(MODEL.split('/')[-1], len(cats), len(constructs)), fontweight="bold")
fig.tight_layout()
slug=MODEL.replace("/","__"); out=f"{SPEC_DRIVE}/{slug}"; os.makedirs(out, exist_ok=True)
for ext in ("png","pdf"): fig.savefig(f"{out}/emotion_spectrum.{ext}", bbox_inches="tight", dpi=200)
json.dump({"model":MODEL,"matrix":M,"emo_vad":EMO_VAD,"vad_regression":reg,
           "pca_explained":pc.explained_variance_ratio_.tolist(),"n_emo":N_EMO,
           "ts":time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())},
          open(f"{out}/emotion_spectrum.json","w"), indent=2, default=float)
print("saved ->", out)
print("\nVAD regression (which dimension drives each behavior):")
for k in constructs: print("  %-8s V%+.2f A%+.2f D%+.2f  R²=%.2f"%(k,reg[k]["V"],reg[k]["A"],reg[k]["D"],reg[k]["R2"]))